In [1]:
#%%writefile unet.py


import torch
import torch.nn as nn 

def double_conv(in_channel, out_channel):
    conv = nn.Sequential(
        nn.Conv2d(in_channel, out_channel, kernel_size=3),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channel, out_channel, kernel_size=3),
        nn.ReLU(inplace=True)
    )
    
    return conv


def crop_img(tensor, target_tensor):
    """
    function to crop image since x and x7 sizes are not the same. 
    """
    # # bs, c , h, w 
    target_size = target_tensor.size()[2]
    tensor_size = tensor.size()[2]
    delta = tensor_size - target_size
    delta = delta // 2 

    return tensor[: , :, delta:tensor_size-delta, delta:tensor_size-delta]


class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        self.max_pool_2x2 = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.down_conv_1 = double_conv(1, 64)
        self.down_conv_2 = double_conv(64, 128)
        self.down_conv_3 = double_conv(128, 256)
        self.down_conv_4 = double_conv(256, 512)
        self.down_conv_5 = double_conv(512, 1024)

        self.up_trans_1 = nn.ConvTranspose2d(in_channels=1024, out_channels=512, kernel_size=2, stride=2)
        self.up_trans_2 = nn.ConvTranspose2d(in_channels=512, out_channels=256, kernel_size=2, stride=2)
        self.up_trans_3 = nn.ConvTranspose2d(in_channels=256, out_channels=128, kernel_size=2, stride=2)
        self.up_trans_4 = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=2, stride=2)

        self.upconv_1 = double_conv(1024, 512)
        self.upconv_2 = double_conv(512, 256)
        self.upconv_3 = double_conv(256, 128)
        self.upconv_4 = double_conv(128, 64)

        self.out = nn.Conv2d(in_channels=64, out_channels=2, kernel_size=1)
    
    def forward(self, image):
        # bs, c , h, w 
        # encoder 
        x1 = self.down_conv_1(image) #
        x2 = self.max_pool_2x2(x1) 
        x3 = self.down_conv_2(x2) #
        x4 = self.max_pool_2x2(x3) 
        x5 = self.down_conv_3(x4) #
        x6 = self.max_pool_2x2(x5) 
        x7 = self.down_conv_4(x6) #
        x8 = self.max_pool_2x2(x7) 
        x9 = self.down_conv_5(x8)  #

        # decode 
        x = self.up_trans_1(x9) 
        y = crop_img(x7, x)
        x = self.upconv_1(torch.cat([x, y],1))
        x = self.up_trans_2(x) 
        y = crop_img(x5, x)
        x = self.upconv_2(torch.cat([x, y],1))
        x = self.up_trans_3(x) 
        y = crop_img(x3, x)
        x = self.upconv_3(torch.cat([x, y],1))
        x = self.up_trans_4(x) 
        y = crop_img(x1, x)
        x = self.upconv_4(torch.cat([x, y],1))
        
        x = self.out(x)

        print(x.size())
        
        return x
        

if __name__ == "__main__":
    image = torch.rand((1,1,572,572))
    model = UNet() 
    print(model(image))


torch.Size([1, 2, 388, 388])
tensor([[[[0.0206, 0.0290, 0.0238,  ..., 0.0235, 0.0179, 0.0208],
          [0.0271, 0.0252, 0.0202,  ..., 0.0146, 0.0198, 0.0231],
          [0.0193, 0.0255, 0.0210,  ..., 0.0237, 0.0231, 0.0177],
          ...,
          [0.0271, 0.0158, 0.0234,  ..., 0.0171, 0.0171, 0.0165],
          [0.0213, 0.0245, 0.0224,  ..., 0.0216, 0.0262, 0.0233],
          [0.0185, 0.0222, 0.0189,  ..., 0.0180, 0.0242, 0.0187]],

         [[0.0697, 0.0695, 0.0749,  ..., 0.0769, 0.0701, 0.0765],
          [0.0729, 0.0746, 0.0699,  ..., 0.0724, 0.0753, 0.0838],
          [0.0699, 0.0727, 0.0711,  ..., 0.0765, 0.0718, 0.0767],
          ...,
          [0.0799, 0.0739, 0.0727,  ..., 0.0784, 0.0755, 0.0787],
          [0.0694, 0.0727, 0.0758,  ..., 0.0677, 0.0810, 0.0739],
          [0.0772, 0.0755, 0.0722,  ..., 0.0735, 0.0749, 0.0650]]]],
       grad_fn=<MkldnnConvolutionBackward>)


In [3]:
#%%writefile dataset.py
import os 
import glob 
import torch 

import numpy as np 
import pandas as pd 

from PIL import Image, ImageFile 

from tqdm import tqdm 
from collections import defaultdict 
from torchvision import transforms 

from albumentations import (Compose, OneOf, RandomBrightnessContrast, RandomGamma, ShiftScaleRotate)
ImageFile.LOAD_TRUNCATED_IMAGES = True 

class SIIMDataset(torch.utils.data.Dataset):
    def __init__(self, image_ids, transform=True, preprocessing_fn=None):
        """
        image_ids: id of images, list 
        transform: True/False
        preprocessing_fn: a function for preprocessing image 
        """

        # empty dict to store image and mask paths 
        self.data = defaultdict(dict)
        #for augmentation 
        self.transform = transform 
        #preprocessing function 
        self.preprocessing_fn = preprocessing_fn

        #argumentation 
        self.aug = Compose([ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=10, p=0.8),
                            OneOf([RandomGamma(gamma_limit=(90,100)), RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1),], p=0.5,),])

        # edit this part to how 
        for imgid in image_ids:
            files = glob.glob(os.path.join(TRAIN_PATH, imgid, "*.png")) 
            self.data[counter] = {"img_path": os.path.join(TRAIN_PATH, imgid, "*.png"),
                                    "mask_path": os.path.join(TRAIN_PATH, imgid, "_mask.png"), }

        def __len__(self):
            return len(self.data) 
        
        def __getitem__(self, item):
            # for a given item index, return image and mask tensor, and read image and mask path 
            img_path = self.data[item]["img_path"]
            mask_path = self.data[item]["mask_path"]

            img = Image.open(img_path)
            img = img.convert("RGB") 
            img = np.array(img) 

            mask = Image.open(mask_path)
            mask = (mask >= 1).astype("float32") # convert to binary float matrix  

            if self.transform is True:
                augmented = self.aug(image=img, mask=mask)
                img = augmented["image"]
                mask = augmented["mask"]

            img = self.preprocessing_fn(img) #normalized image 

            return {"image": transforms.ToTensor()(img),
                    "mask": transforms.ToTensor()(mask).float(),}
